# Extração de Features
## Parte 2: Extração Propriamente Dita — Entregável 6

Este é o **notebook central** da disciplina. Opera sobre o array 3D segmentado produzido
pelo Entregável 5 e extrai 4 famílias de features por janela:

| Família | Canais-alvo | N aprox. |
|---------|-------------|----------|
| Domínio do tempo | Todos (54) | 291 |
| Domínio da frequência | EEG, EMG, Bio, IMU | 290 |
| Tempo-frequência (STFT + DWT) | EEG, EMG | 661 |
| Não-lineares | EEG-C3, EEG-FP1, EMG-RTA | 9 |

**Total: ~1.251 features por janela.**

### Entrada
- `data/silver/segments.npz` → `X` (24 564, 500, 54), `y` (24 564,), `canais`
- `data/silver/windows_metadata.parquet` → metadados por janela

### Saída
- `data/silver/features_raw.parquet` → DataFrame (n_janelas, n_features + 3 colunas de meta)

### Convenção de nomenclatura
```
{canal}__{domínio}__{métrica}
Ex: EEG-C3__freq__band_alpha
    EMG-RTA__time__RMS
    EEG-FP1__tfreq__stft_beta_t2
    EEG-CZ__tfreq__dwt_D4_energy
    EEG-C3__nonlin__sample_entropy
```

## 0. Imports e constantes

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import signal as scsig
from scipy import stats as sts
from scipy.spatial import cKDTree
import pywt
import time
import warnings
warnings.filterwarnings('ignore')

# ── Caminhos ──────────────────────────────────────────────────────────────────
ROOT          = Path('..')
SEGMENTS_PATH = ROOT / 'data' / 'silver' / 'segments.npz'
META_PATH     = ROOT / 'data' / 'silver' / 'windows_metadata.parquet'
FEATURES_PATH = ROOT / 'data' / 'silver' / 'features_raw.parquet'

# ── Parâmetros globais ────────────────────────────────────────────────────────
FS       = 500
WINDOW_N = 500

# ── Listas de canais por tipo ─────────────────────────────────────────────────
EEG_COLS = [
    'EEG-FP1','EEG-FP2','EEG-F3','EEG-F4','EEG-C3','EEG-C4',
    'EEG-P3','EEG-P4','EEG-O1','EEG-O2','EEG-F7','EEG-F8',
    'EEG-P7','EEG-P8','EEG-FZ','EEG-CZ','EEG-PZ',
    'EEG-FC1','EEG-FC2','EEG-CP1','EEG-CP2',
    'EEG-FC5','EEG-FC6','EEG-CP5','EEG-CP6'
]
EMG_COLS   = ['EMG-RTA', 'EMG-LTA', 'EMG-RGS']
BIO_OUTROS = ['IO', 'ECG']
SENSOR_NAMES  = ['LShank', 'RShank', 'Waist', 'Arm']
IMU_SINAL_SUF = ['ACCX','ACCY','ACCZ','GYRO-X','GYRO-Y','GYRO-Z']
IMU_COLS = [f'{s}-{ax}' for s in SENSOR_NAMES for ax in IMU_SINAL_SUF]

# ── Bandas de frequência ──────────────────────────────────────────────────────
EEG_BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
    'gamma': (30.0, 40.0),
}
EMG_BANDS = {
    'low' : ( 20.0,  50.0),
    'mid' : ( 50.0, 150.0),
    'high': (150.0, 200.0),
}
IMU_BANDS = {
    'movimento': (0.1,  3.0),
    'tremor'  : (3.0, 10.0),
}

# ── Parâmetros STFT ───────────────────────────────────────────────────────────
# 250ms de sub-janela @ 500Hz = 125 amostras → 4 bins de tempo sem sobreposição
# Resolução em frequência resultante: 500 / 125 = 4 Hz
STFT_NPERSEG  = 125
STFT_NOVERLAP = 0
N_STFT_TBINS  = WINDOW_N // STFT_NPERSEG  # = 4

# ── Parâmetros DWT ────────────────────────────────────────────────────────────
DWT_WAVELET = 'db4'
DWT_LEVELS  = 4

# ── Canais para features não-lineares (computacionalmente caras) ──────────────
CANAIS_NONLIN = ['EEG-C3', 'EEG-FP1', 'EMG-RTA']

print(f'FS = {FS} Hz  |  janela = {WINDOW_N} amostras ({WINDOW_N/FS}s)')
print(f'STFT: sub-janelas de {STFT_NPERSEG} amostras ({STFT_NPERSEG/FS*1000:.0f}ms) → {N_STFT_TBINS} bins de tempo')
print(f'DWT : {DWT_WAVELET}, {DWT_LEVELS} níveis  (D1:125-250Hz, D2:62.5-125Hz, D3:31.2-62.5Hz, D4:15.6-31.2Hz, A4:0-15.6Hz)')
print(f'Canais não-lineares: {CANAIS_NONLIN}')

FS = 500 Hz  |  janela = 500 amostras (1.0s)
STFT: sub-janelas de 125 amostras (250ms) → 4 bins de tempo
DWT : db4, 4 níveis  (D1:125-250Hz, D2:62.5-125Hz, D3:31.2-62.5Hz, D4:15.6-31.2Hz, A4:0-15.6Hz)
Canais não-lineares: ['EEG-C3', 'EEG-FP1', 'EMG-RTA']


## 1. Carregar dados segmentados

In [2]:
print('Carregando segments.npz...')
dados  = np.load(SEGMENTS_PATH, allow_pickle=True)
X      = dados['X']              # (n_janelas, 500, 54) float32
y      = dados['y']              # (n_janelas,) float32
canais = dados['canais'].tolist()

df_meta = pd.read_parquet(META_PATH)

N_WIN, N_AMS, N_CH = X.shape

# Índices globais de cada categoria de canal
idx_eeg = [canais.index(c) for c in EEG_COLS  if c in canais]
idx_emg = [canais.index(c) for c in EMG_COLS  if c in canais]
idx_bio = [canais.index(c) for c in BIO_OUTROS if c in canais]
idx_imu = [canais.index(c) for c in IMU_COLS  if c in canais]

eeg_nomes = [canais[i] for i in idx_eeg]
emg_nomes = [canais[i] for i in idx_emg]
bio_nomes = [canais[i] for i in idx_bio]
imu_nomes = [canais[i] for i in idx_imu]

print(f'X      : {X.shape}  dtype={X.dtype}')
print(f'y      : label=0 → {(y==0).sum():,}  |  label=1 → {(y==1).sum():,}  |  NaN → {np.isnan(y).sum()}')
print(f'Canais : {N_CH} total  ({len(eeg_nomes)} EEG | {len(emg_nomes)} EMG | {len(bio_nomes)} Bio | {len(imu_nomes)} IMU)')
print()
nan_pct_canal = np.isnan(X).mean(axis=(0,1)) * 100
canais_com_nan = [(canais[i], nan_pct_canal[i]) for i in range(N_CH) if nan_pct_canal[i] > 0]
print(f'Canais com NaN (sensores ausentes em alguns pacientes): {len(canais_com_nan)}')
for nome, pct in sorted(canais_com_nan, key=lambda x: -x[1])[:8]:
    print(f'  {nome:<22}: {pct:.1f}% NaN')
if len(canais_com_nan) > 8:
    print(f'  ... e mais {len(canais_com_nan)-8} canais')

Carregando segments.npz...
X      : (24564, 500, 54)  dtype=float32
y      : label=0 → 14,386  |  label=1 → 10,178  |  NaN → 0
Canais : 54 total  (25 EEG | 3 EMG | 2 Bio | 24 IMU)

Canais com NaN (sensores ausentes em alguns pacientes): 24
  RShank-ACCX           : 53.3% NaN
  RShank-ACCY           : 53.3% NaN
  RShank-ACCZ           : 53.3% NaN
  RShank-GYRO-X         : 53.3% NaN
  RShank-GYRO-Y         : 53.3% NaN
  RShank-GYRO-Z         : 53.3% NaN
  Waist-ACCX            : 44.9% NaN
  Waist-ACCY            : 44.9% NaN
  ... e mais 16 canais


## 2. Passo 1 — Domínio do Tempo

Features calculadas sobre os 500 pontos de cada janela por canal.

| Feature | Fórmula | Interpretação |
|---------|---------|---------------|
| **RMS** | √(mean(x²)) | Energia do sinal; para EMG correlaciona com força muscular |
| **MAV** | mean(|x|) | Mais robusto ao ruído que RMS em alguns contextos |
| **VAR** | var(x) | Complementar ao std; importante quando média ≠ 0 |
| **ZCR** | cruzamentos/s | Para EMG: nível de ativação; para EEG: ritmos rápidos |
| **Hjorth Activity** | var(x) | Energia na janela (EEG apenas) |
| **Hjorth Mobility** | √(var(dx)/var(x)) | Frequência dominante normalizada (EEG) |
| **Hjorth Complexity** | mobility(dx)/mobility(x) | Complexidade do formato de onda (EEG) |

Implementação **totalmente vetorizada** — processa todos os canais e todas as janelas de uma vez.

In [ ]:
print('Calculando features de domínio do tempo (vetorizado)...')
t0_total = time.time()
feat_time = {}

# ── RMS ───────────────────────────────────────────────────────────────────────
rms = np.sqrt(np.nanmean(X ** 2, axis=1))          # (N_WIN, N_CH)
for i, col in enumerate(canais):
    feat_time[f'{col}__time__RMS'] = rms[:, i]

# ── MAV ───────────────────────────────────────────────────────────────────────
mav = np.nanmean(np.abs(X), axis=1)                # (N_WIN, N_CH)
for i, col in enumerate(canais):
    feat_time[f'{col}__time__MAV'] = mav[:, i]

# ── Variância ─────────────────────────────────────────────────────────────────
var = np.nanvar(X, axis=1)                          # (N_WIN, N_CH), ddof=0
for i, col in enumerate(canais):
    feat_time[f'{col}__time__VAR'] = var[:, i]

# ── ZCR ───────────────────────────────────────────────────────────────────────
# NaN → 0 antes do sign para evitar propagação (NaN != 0 seria True → falso cruzamento)
X_zcr = np.where(np.isnan(X), 0.0, X)
sgn   = np.sign(X_zcr).astype(np.float32)
sgn[sgn == 0] = 1.0                                 # amostras exatamente zero → +1
zcr   = np.sum(np.diff(sgn, axis=1) != 0, axis=1) / (N_AMS / FS)  # (N_WIN, N_CH)
for i, col in enumerate(canais):
    feat_time[f'{col}__time__ZCR'] = zcr[:, i]
del sgn, X_zcr

n_feat_basicas = 4 * N_CH
print(f'  RMS, MAV, VAR, ZCR → {n_feat_basicas} features ({N_CH} canais × 4)')

# ── Parâmetros de Hjorth (apenas EEG) ────────────────────────────────────────
X_eeg = X[:, :, idx_eeg].astype(np.float64)        # (N_WIN, 500, n_eeg)
n_eeg = len(idx_eeg)

var_x  = np.nanvar(X_eeg,              axis=1) + 1e-12  # (N_WIN, n_eeg)
dx     = np.diff(X_eeg, n=1, axis=1)                   # (N_WIN, 499, n_eeg)
d2x    = np.diff(X_eeg, n=2, axis=1)                   # (N_WIN, 498, n_eeg)
var_dx  = np.nanvar(dx,  axis=1) + 1e-12
var_d2x = np.nanvar(d2x, axis=1) + 1e-12

hjorth_activity   = np.nanvar(X_eeg, axis=1)           # = var_x - 1e-12
hjorth_mobility   = np.sqrt(var_dx   / var_x)
hjorth_complexity = np.sqrt(var_d2x  / var_dx) / (hjorth_mobility + 1e-12)

for i, col in enumerate(eeg_nomes):
    feat_time[f'{col}__time__hjorth_activity']   = hjorth_activity[:, i]
    feat_time[f'{col}__time__hjorth_mobility']   = hjorth_mobility[:, i]
    feat_time[f'{col}__time__hjorth_complexity'] = hjorth_complexity[:, i]

del X_eeg, dx, d2x, var_x, var_dx, var_d2x
n_hjorth = 3 * n_eeg
n_feat_time = n_feat_basicas + n_hjorth
print(f'  Hjorth → {n_hjorth} features ({n_eeg} canais EEG × 3)')
print(f'  Total domínio do tempo: {n_feat_time} features  ({time.time()-t0_total:.1f}s)')

Calculando features de domínio do tempo (vetorizado)...
  RMS, MAV, VAR, ZCR → 216 features (54 canais × 4)


## 3. Passo 2 — Domínio da Frequência

PSD via **Welch** com `nperseg = 500` (janela inteira) → resolução espectral de **1 Hz**.
Como cada janela tem exatamente 500 amostras, isso equivale a um único periodograma.

| Tipo | Bandas | Outras features |
|------|--------|-----------------|
| EEG  | delta 0.5–4, theta 4–8, alpha 8–13, beta 13–30, gamma 30–40 Hz | freq. mediana, freq. de pico |
| EMG  | low 20–50, mid 50–150, high 150–200 Hz | freq. mediana, freq. de pico |
| IMU  | movimento 0.1–3, tremor 3–10 Hz | freq. mediana, freq. de pico |
| Bio (IO, ECG) | — | freq. mediana, freq. de pico |

> **Nota NaN**: canais IMU com NaN (sensor ausente em alguns pacientes) são tratados
> com interpolação linear antes do Welch. Janelas 100% NaN produzem features NaN.

In [ ]:
def _interp_nan_rows(X_ch):
    """Para cada janela com NaN, interpola linearmente. Retorna cópia float32."""
    out = X_ch.astype(np.float32, copy=True)
    nan_mask = np.isnan(out)
    rows_com_nan = np.where(nan_mask.any(axis=1))[0]
    for i in rows_com_nan:
        row = out[i].astype(np.float64)
        nans = np.isnan(row)
        ok   = ~nans
        if ok.any():
            row[nans] = np.interp(np.where(nans)[0], np.where(ok)[0], row[ok])
            out[i] = row.astype(np.float32)
        else:
            out[i] = 0.0   # canal 100% NaN → zeros (features ficarão NaN via PSD=0)
    return out


def welch_features(X_ch_raw, ch_name, bands):
    """Calcula PSD via Welch e extrai band power, freq_mediana, freq_pico.
    X_ch_raw: (N_WIN, 500) — canal único, pode ter NaN.
    Retorna dict feature_name → np.array(N_WIN,).
    """
    X_ch = _interp_nan_rows(X_ch_raw)
    # Marca janelas que eram 100% NaN para propagar NaN nas features
    all_nan_rows = np.isnan(X_ch_raw).all(axis=1)  # antes da interpolação

    freqs_w, psd = scsig.welch(X_ch, fs=FS, nperseg=N_AMS,
                                axis=1, scaling='density')
    # psd: (N_WIN, 251)  freqs_w: [0, 1, ..., 250] Hz

    out = {}
    for band_name, (f_lo, f_hi) in bands.items():
        mask  = (freqs_w >= f_lo) & (freqs_w <= f_hi)
        bp    = np.trapz(psd[:, mask], freqs_w[mask], axis=1) if mask.sum() >= 2 \
                else np.sum(psd[:, mask], axis=1)
        bp[all_nan_rows] = np.nan
        out[f'{ch_name}__freq__band_{band_name}'] = bp

    # Frequência mediana: freq abaixo da qual está 50% da potência total
    cumsum   = np.cumsum(psd, axis=1)          # (N_WIN, 251)
    total    = cumsum[:, -1:]                  # (N_WIN, 1)
    idx_med  = np.argmax(cumsum >= total / 2.0, axis=1)  # (N_WIN,)
    med_freq = freqs_w[idx_med].astype(np.float32)
    med_freq[all_nan_rows] = np.nan
    out[f'{ch_name}__freq__median_freq'] = med_freq

    # Frequência de pico
    peak_freq = freqs_w[np.argmax(psd, axis=1)].astype(np.float32)
    peak_freq[all_nan_rows] = np.nan
    out[f'{ch_name}__freq__peak_freq'] = peak_freq

    return out


print('Calculando features de domínio da frequência...')
t0_freq = time.time()
feat_freq = {}

# ── EEG ──────────────────────────────────────────────────────────────────────
print(f'  EEG ({n_eeg} canais)...')
for ch_idx_local, ch_name in enumerate(eeg_nomes):
    feat_freq.update(welch_features(X[:, :, idx_eeg[ch_idx_local]], ch_name, EEG_BANDS))
n_feat_freq_eeg = n_eeg * (len(EEG_BANDS) + 2)
print(f'    {n_eeg} × {len(EEG_BANDS)+2} = {n_feat_freq_eeg} features')

# ── EMG ──────────────────────────────────────────────────────────────────────
print(f'  EMG ({len(emg_nomes)} canais)...')
for ch_name in emg_nomes:
    feat_freq.update(welch_features(X[:, :, canais.index(ch_name)], ch_name, EMG_BANDS))
n_feat_freq_emg = len(emg_nomes) * (len(EMG_BANDS) + 2)
print(f'    {len(emg_nomes)} × {len(EMG_BANDS)+2} = {n_feat_freq_emg} features')

# ── Bio (IO, ECG) — apenas freq mediana e pico ───────────────────────────────
for ch_name in bio_nomes:
    feat_freq.update(welch_features(X[:, :, canais.index(ch_name)], ch_name, {}))
n_feat_freq_bio = len(bio_nomes) * 2

# ── IMU ──────────────────────────────────────────────────────────────────────
print(f'  IMU ({len(imu_nomes)} canais)...')
for ch_name in imu_nomes:
    feat_freq.update(welch_features(X[:, :, canais.index(ch_name)], ch_name, IMU_BANDS))
n_feat_freq_imu = len(imu_nomes) * (len(IMU_BANDS) + 2)
print(f'    {len(imu_nomes)} × {len(IMU_BANDS)+2} = {n_feat_freq_imu} features')

n_feat_freq = len(feat_freq)
print(f'  Total domínio da frequência: {n_feat_freq} features  ({time.time()-t0_freq:.1f}s)')

## 4. Passo 3 — Tempo-Frequência

### 4.1 STFT — Short-Time Fourier Transform

Para cada janela de 1s, dividimos em **4 sub-janelas de 250ms** (125 amostras) sem sobreposição.
Para cada sub-janela, calculamos a potência espectral em cada banda.

```
Janela (1s = 500 amostras)
 ┌──────────────────────────────────────────────────┐
 │   t0 [0:125]  │  t1 [125:250] │  t2 [250:375] │  t3 [375:500] │
 └──────────────────────────────────────────────────┘
   250ms            250ms           250ms           250ms
```

> **Limitação conhecida**: resolução em frequência = 500/125 = **4 Hz**.
> Isso é suficiente para detectar a distribuição de energia entre bandas largas,
> mas não distingue com precisão bandas estreitas como delta (0.5–4 Hz) ou theta (4–8 Hz).
> O objetivo aqui é capturar a **evolução temporal** da energia dentro da janela de 1s.

Aplicado a: **EEG** (5 bandas × 4 bins = 20 features/canal) e **EMG** (3 bandas × 4 bins = 12 features/canal).

In [ ]:
def stft_band_powers(X_ch_raw, ch_name, bands):
    """STFT batch para todas as janelas de um canal.
    Retorna dict: '{canal}__tfreq__stft_{banda}_t{k}' → array(N_WIN,).
    """
    X_ch = np.nan_to_num(X_ch_raw, nan=0.0).astype(np.float32)
    all_nan_rows = np.isnan(X_ch_raw).all(axis=1)

    # scipy.signal.stft com axis=1 processa todas as janelas de uma vez
    _, _, Zxx = scsig.stft(
        X_ch, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP,
        axis=1, boundary=None, padded=False
    )
    # Zxx: (N_WIN, n_freqs, n_tbins)  com n_freqs = 63, n_tbins = 4
    psd_s = np.abs(Zxx) ** 2          # potência instantânea
    freqs_s = np.fft.rfftfreq(STFT_NPERSEG, d=1.0 / FS)  # [0, 4, 8, ..., 248]

    out = {}
    for band_name, (f_lo, f_hi) in bands.items():
        mask = (freqs_s >= f_lo) & (freqs_s <= f_hi)
        for t_idx in range(N_STFT_TBINS):
            vals = np.sum(psd_s[:, mask, t_idx], axis=1).astype(np.float32)
            vals[all_nan_rows] = np.nan
            out[f'{ch_name}__tfreq__stft_{band_name}_t{t_idx}'] = vals
    return out


print('Calculando features STFT...')
t0_stft = time.time()
feat_stft = {}

print(f'  EEG ({n_eeg} canais × {len(EEG_BANDS)} bandas × {N_STFT_TBINS} bins)...')
for ch_idx_local, ch_name in enumerate(eeg_nomes):
    feat_stft.update(stft_band_powers(X[:, :, idx_eeg[ch_idx_local]], ch_name, EEG_BANDS))

n_stft_per_eeg = len(EEG_BANDS) * N_STFT_TBINS
n_stft_per_emg = len(EMG_BANDS) * N_STFT_TBINS

print(f'  EMG ({len(emg_nomes)} canais × {len(EMG_BANDS)} bandas × {N_STFT_TBINS} bins)...')
for ch_name in emg_nomes:
    feat_stft.update(stft_band_powers(X[:, :, canais.index(ch_name)], ch_name, EMG_BANDS))

n_feat_stft = len(feat_stft)
print(f'  Total STFT: {n_feat_stft} features  ({time.time()-t0_stft:.1f}s)')
print(f'    EEG: {n_eeg} × {n_stft_per_eeg} = {n_eeg * n_stft_per_eeg}')
print(f'    EMG: {len(emg_nomes)} × {n_stft_per_emg} = {len(emg_nomes) * n_stft_per_emg}')

### 4.2 DWT — Discrete Wavelet Transform (Daubechies 4)

A DWT decompõe o sinal em níveis de detalhe e aproximação.
Para `db4` com 4 níveis a 500 Hz, o mapeamento aproximado de frequência é:

| Coeficiente | Banda (Hz) | Relevância EEG |
|-------------|------------|----------------|
| D1 | 125 – 250 | Ruído de alta frequência (deve ter energia baixa em EEG filtrado) |
| D2 | 62.5 – 125 | Ruído / muscle artifact |
| D3 | 31.25 – 62.5 | Alta beta / gamma |
| D4 | 15.6 – 31.25 | Beta |
| A4 | 0 – 15.6 | Delta + theta + alpha |

Feature extraída: **energia normalizada** = Σ(coeff²) / len(coeff) — invariante ao comprimento do nível.

Aplicado a: **todos os 25 canais EEG** (5 coeficientes × 25 canais = 125 features).

In [ ]:
print('Calculando features DWT (db4, 4 níveis, somente EEG)...')
t0_dwt = time.time()
feat_dwt = {}

# Nomes das features por nível (D1..D4 + A4)
_dwt_labels = [f'D{lv}' for lv in range(1, DWT_LEVELS + 1)] + [f'A{DWT_LEVELS}']
n_dwt_per_canal = len(_dwt_labels)   # = 5

# Pré-alocar arrays
for ch_name in eeg_nomes:
    for lbl in _dwt_labels:
        feat_dwt[f'{ch_name}__tfreq__dwt_{lbl}_energy'] = np.full(N_WIN, np.nan, dtype=np.float32)

for ch_idx_local, ch_name in enumerate(eeg_nomes):
    x_ch = X[:, :, idx_eeg[ch_idx_local]]   # (N_WIN, 500)

    for win_i in range(N_WIN):
        sinal = x_ch[win_i].astype(np.float64)

        if np.all(np.isnan(sinal)):
            continue   # deixa NaN

        # Interpolar NaN residuais (EEG não tem NaN, mas por robustez)
        if np.any(np.isnan(sinal)):
            nans = np.isnan(sinal)
            ok   = ~nans
            sinal[nans] = np.interp(np.where(nans)[0], np.where(ok)[0], sinal[ok])

        # wavedec: coeffs[0]=A4, coeffs[1]=D4, ..., coeffs[4]=D1
        coeffs = pywt.wavedec(sinal, DWT_WAVELET, level=DWT_LEVELS)

        # D1..D4 (energia normalizada por comprimento)
        for lv in range(1, DWT_LEVELS + 1):
            det = coeffs[DWT_LEVELS + 1 - lv]   # D1=coeffs[4], D4=coeffs[1]
            feat_dwt[f'{ch_name}__tfreq__dwt_D{lv}_energy'][win_i] = \
                np.sum(det ** 2) / max(len(det), 1)

        # A4 (aproximação)
        ap = coeffs[0]
        feat_dwt[f'{ch_name}__tfreq__dwt_A{DWT_LEVELS}_energy'][win_i] = \
            np.sum(ap ** 2) / max(len(ap), 1)

    if (ch_idx_local + 1) % 5 == 0 or ch_idx_local == n_eeg - 1:
        print(f'  {ch_idx_local+1}/{n_eeg} canais  ({time.time()-t0_dwt:.0f}s)')

n_feat_dwt = len(feat_dwt)
print(f'Total DWT: {n_feat_dwt} features  ({time.time()-t0_dwt:.1f}s)')
print(f'  {n_eeg} canais × {n_dwt_per_canal} níveis = {n_eeg * n_dwt_per_canal}')

## 5. Passo 4 — Features Não-Lineares

Features que capturam complexidade e auto-correlação do sinal, **não deriváveis** via
transformadas lineares. São computacionalmente mais caras (O(N²) ou O(N log N) por janela).

| Feature | Algoritmo | O() | Interpretação |
|---------|-----------|-----|---------------|
| **SampEn** | KD-Tree Chebyshev | O(N log N) | Regularidade: baixo = sinal mecânico/artefato; alto = fisiológico |
| **ApEn** | KD-Tree Chebyshev | O(N log N) | Versão com auto-comparação; inclui pontos de borda |
| **DFA α** | Vetorizado por escala | O(N·S) | Expoente fractal; EEG saudável: 0.5 < α < 1 |

**Parâmetros**: m=2, r = 0.2 × std(janela).

**Aplicado apenas a 3 canais chave**: `EEG-C3`, `EEG-FP1`, `EMG-RTA`.
Os demais canais seriam igualmente válidos, mas o custo computacional por
canal adicional é ~2–4 minutos a mais.

In [ ]:
# ── Funções não-lineares ──────────────────────────────────────────────────────

def _prep(x):
    """Float64 + interpolação linear de NaN."""
    x = np.asarray(x, dtype=np.float64)
    if np.any(np.isnan(x)):
        nans = np.isnan(x)
        ok   = ~nans
        if ok.any():
            x[nans] = np.interp(np.where(nans)[0], np.where(ok)[0], x[ok])
        else:
            return None
    return x


def sample_entropy(x, m=2, r_factor=0.2):
    """Sample Entropy via KD-Tree com distância Chebyshev (O(N log N)).
    Não conta auto-correspondências (i = j).
    Retorna NaN se r < 1e-10 ou B = 0.
    """
    x = _prep(x)
    if x is None:
        return np.nan
    r = r_factor * np.std(x)
    if r < 1e-10:
        return np.nan
    N = len(x)
    # Templates de comprimento m e m+1 (sem a última posição para alinhar com m+1)
    Tm  = np.lib.stride_tricks.sliding_window_view(x, m  )[: N - m - 1]  # (N-m-1, m)
    Tm1 = np.lib.stride_tricks.sliding_window_view(x, m+1)[: N - m - 1]  # (N-m-1, m+1)
    tree_m   = cKDTree(Tm)
    tree_m1  = cKDTree(Tm1)
    # count_neighbors conta pares ordenados (i, j), incluindo (i, i); subtrair self-matches
    B = tree_m.count_neighbors(tree_m,   r=r, p=np.inf) - len(Tm)
    A = tree_m1.count_neighbors(tree_m1, r=r, p=np.inf) - len(Tm1)
    if B <= 0:
        return np.nan
    if A <= 0:
        return np.inf
    return float(-np.log(A / B))


def approx_entropy(x, m=2, r_factor=0.2):
    """Approximate Entropy via KD-Tree. Inclui auto-correspondências (i = j).
    phi(m) = mean(log(C_i^m / N_m)) onde C_i conta incluindo self-match.
    """
    x = _prep(x)
    if x is None:
        return np.nan
    r = r_factor * np.std(x)
    if r < 1e-10:
        return np.nan
    N = len(x)

    def phi(m_len):
        Tm = np.lib.stride_tricks.sliding_window_view(x, m_len)  # (N-m_len+1, m_len)
        if len(Tm) == 0:
            return 0.0
        tree = cKDTree(Tm)
        # Para cada ponto, conta vizinhos (inclusive self) dentro de raio r
        neighbors = tree.query_ball_tree(tree, r=r, p=np.inf)
        counts = np.array([len(nb) for nb in neighbors], dtype=np.float64)
        counts = np.clip(counts, 1, None)        # evita log(0)
        return float(np.mean(np.log(counts / len(Tm))))

    return float(phi(m) - phi(m + 1))


def dfa_exponent(x, min_n=10, n_scales=10):
    """DFA vetorizado: retorna expoente de escala alfa.
    EEG típico: 0.5 < alfa < 1 (correlações de longo alcance).
    Ruído branco: alfa ≈ 0.5; movimento browniano: alfa ≈ 1.5.
    """
    x = _prep(x)
    if x is None:
        return np.nan
    N     = len(x)
    max_n = N // 4
    y_int = np.cumsum(x - np.mean(x))   # sinal integrado

    scales = np.unique(
        np.round(np.logspace(np.log10(min_n), np.log10(max_n), n_scales)).astype(int)
    )
    flucts = []
    valid  = []
    _H_cache = {}   # cache das matrizes de detrending por escala

    for n in scales:
        if n < 4 or n > N // 2:
            continue
        n_segs = N // n
        if n_segs < 2:
            continue

        if n not in _H_cache:
            t = np.arange(n, dtype=np.float64)
            A = np.column_stack([np.ones(n), t])          # (n, 2)
            # H = I - projeção linear: (I - A(A'A)^{-1}A') = residual de fit linear
            _H_cache[n] = np.eye(n) - A @ np.linalg.solve(A.T @ A, A.T)

        H    = _H_cache[n]
        segs = y_int[: n_segs * n].reshape(n_segs, n)    # (n_segs, n)
        resid = segs @ H.T                                # (n_segs, n), H simétrico
        F_n   = np.sqrt(np.mean(resid ** 2))
        flucts.append(F_n)
        valid.append(n)

    if len(valid) < 3:
        return np.nan
    alfa, _ = np.polyfit(
        np.log10(valid),
        np.log10(np.clip(flucts, 1e-10, None)),
        1
    )
    return float(alfa)


# Benchmark rápido (5 janelas) para estimar tempo total
np.random.seed(0)
_bench = [np.random.randn(N_AMS) for _ in range(5)]
t0_b = time.time()
for s in _bench:
    sample_entropy(s); approx_entropy(s); dfa_exponent(s)
dt_b = (time.time() - t0_b) / 5
n_computacoes = N_WIN * len(CANAIS_NONLIN) * 3   # 3 features
print(f'Benchmark: {dt_b*1000:.1f}ms / (janela × 3 features)')
print(f'Estimativa total: {n_computacoes * dt_b / 60:.0f} min  ({N_WIN:,} janelas × {len(CANAIS_NONLIN)} canais × 3 métricas)')

In [ ]:
print('Calculando features não-lineares (pode demorar ~8 min)...')
print('Canais:', CANAIS_NONLIN)
print()

feat_nonlin = {}
canais_nl_presentes = [c for c in CANAIS_NONLIN if c in canais]

for ch_name in canais_nl_presentes:
    feat_nonlin[f'{ch_name}__nonlin__sample_entropy'] = np.full(N_WIN, np.nan, dtype=np.float32)
    feat_nonlin[f'{ch_name}__nonlin__approx_entropy'] = np.full(N_WIN, np.nan, dtype=np.float32)
    feat_nonlin[f'{ch_name}__nonlin__dfa']            = np.full(N_WIN, np.nan, dtype=np.float32)

t0_nl = time.time()
for ch_n, ch_name in enumerate(canais_nl_presentes):
    ch_idx_global = canais.index(ch_name)
    x_ch = X[:, :, ch_idx_global]   # (N_WIN, 500)
    print(f'  [{ch_n+1}/{len(canais_nl_presentes)}] {ch_name}...', end='', flush=True)
    t0_ch = time.time()

    for win_i in range(N_WIN):
        sinal = x_ch[win_i]
        feat_nonlin[f'{ch_name}__nonlin__sample_entropy'][win_i] = sample_entropy(sinal)
        feat_nonlin[f'{ch_name}__nonlin__approx_entropy'][win_i] = approx_entropy(sinal)
        feat_nonlin[f'{ch_name}__nonlin__dfa'][win_i]            = dfa_exponent(sinal)

    dt_ch = time.time() - t0_ch
    print(f'  {dt_ch:.0f}s  ({dt_ch/N_WIN*1000:.1f}ms/janela)')

n_feat_nonlin = len(feat_nonlin)
print(f'Total não-lineares: {n_feat_nonlin} features  ({time.time()-t0_nl:.1f}s)')

## 6. Passo 5 — Compilar Dataset de Features

Unimos todos os dicts de features em um único DataFrame onde:
- Cada **linha** = uma janela (alinhada com `X[i]` e `y[i]`)
- Cada **coluna** = uma feature nomeada como `{canal}__{domínio}__{métrica}`
- Primeiras 3 colunas: `patient_id`, `task_label`, `janela_inicio_s`

`janela_inicio_s` = `n_inicio / 500`, onde `n_inicio` é o índice de início dentro da
task (conforme `windows_metadata.parquet`). Isso permite rastrear a posição temporal
da janela dentro do sinal original.

In [ ]:
print('Compilando dataset de features...')
t0_compile = time.time()

# Merge de todos os dicts (cada valor é array de shape (N_WIN,))
todas_feats = {}
todas_feats.update(feat_time)
todas_feats.update(feat_freq)
todas_feats.update(feat_stft)
todas_feats.update(feat_dwt)
todas_feats.update(feat_nonlin)

# Verificar alinhamento
for nome, arr in todas_feats.items():
    assert len(arr) == N_WIN, f'Desalinhamento: {nome} tem {len(arr)} != {N_WIN}'

df_features = pd.DataFrame(todas_feats)

# Inserir colunas de metadados no início
df_features.insert(0, 'patient_id',     df_meta['patient_id'].values)
df_features.insert(1, 'task_label',     y.astype(np.float32))
df_features.insert(2, 'janela_inicio_s', (df_meta['n_inicio'].values / FS).astype(np.float32))

n_meta    = 3
n_features = df_features.shape[1] - n_meta

print(f'DataFrame compilado em {time.time()-t0_compile:.1f}s')
print(f'Shape: {df_features.shape}  ({N_WIN:,} janelas × {n_features} features + {n_meta} meta)')
print()
print('Contagem por família:')
fams = {'time': feat_time, 'freq': feat_freq, 'stft': feat_stft, 'dwt': feat_dwt, 'nonlin': feat_nonlin}
total = 0
for nome, d in fams.items():
    print(f'  {nome:<10}: {len(d):>5,} features')
    total += len(d)
print(f'  {"TOTAL":<10}: {total:>5,} features')
print()
print('Primeiras 3 colunas de metadados:')
print(df_features[['patient_id','task_label','janela_inicio_s']].head(3).to_string(index=False))

In [ ]:
print('Salvando features_raw.parquet...')
df_features.to_parquet(FEATURES_PATH, index=False)
size_mb = FEATURES_PATH.stat().st_size / 1e6
print(f'  ✅ {FEATURES_PATH}  ({size_mb:.1f} MB)')
print(f'  Janelas  : {N_WIN:,}')
print(f'  Features : {n_features:,}')
print(f'  Dtype    : {df_features.dtypes.value_counts().to_dict()}')

## 7. Validação do Dataset de Features

In [ ]:
# Recarregar para verificar integridade
df_check = pd.read_parquet(FEATURES_PATH)
feat_cols = [c for c in df_check.columns if c not in ('patient_id','task_label','janela_inicio_s')]

print('=== Validação do dataset de features ===')
print(f'Shape recarregado: {df_check.shape}  ← esperado ({N_WIN}, {n_features+3})')
assert df_check.shape == df_features.shape, '❌ Shape diverge!'
print(f'Shape OK ✅')
print()

# NaN por família
print('--- NaN por família ---')
for nome, d in fams.items():
    cols_fam = [c for c in feat_cols if f'__{nome.replace("stft","tfreq").replace("dwt","tfreq")}__' in c \
                or (nome == 'time'   and '__time__'   in c)
                or (nome == 'freq'   and '__freq__'   in c)
                or (nome == 'nonlin' and '__nonlin__' in c)]
    # Usar a forma simples: contar pelas chaves do dict
    cols_real = [c for c in feat_cols if c in d]
    if not cols_real:
        continue
    nan_total = df_check[cols_real].isna().sum().sum()
    nan_pct   = nan_total / (len(cols_real) * N_WIN) * 100
    print(f'  {nome:<10}: {nan_pct:5.1f}% NaN  ({nan_total:,} valores)')

print()
# Features com NaN > 50%
nan_por_feat = df_check[feat_cols].isna().mean() * 100
feat_muitos_nan = nan_por_feat[nan_por_feat > 50]
if len(feat_muitos_nan) > 0:
    print(f'Features com > 50% NaN: {len(feat_muitos_nan)}')
    print('  (esperado: canais IMU com sensor ausente em vários pacientes)')
    # Mostrar apenas tipos únicos de canal
    canais_afetados = set(f.split('__')[0] for f in feat_muitos_nan.index)
    print(f'  Canais afetados: {sorted(canais_afetados)}')
else:
    print('Nenhuma feature com > 50% NaN ✅')

print()
# Verificar range de valores para EEG-C3 (esperamos Z-scored ≈ [-5, 5])
c3_cols = [c for c in feat_cols if 'EEG-C3__' in c and '__time__' in c]
print('--- Sanidade EEG-C3 features de tempo ---')
for col in c3_cols:
    vals = df_check[col].dropna()
    print(f'  {col:<45}: median={vals.median():.3f}  IQR=[{vals.quantile(0.25):.3f}, {vals.quantile(0.75):.3f}]')

# Verificar features não-lineares
print()
print('--- Sanidade features não-lineares (EEG-C3) ---')
for nl_feat in ['EEG-C3__nonlin__sample_entropy','EEG-C3__nonlin__approx_entropy','EEG-C3__nonlin__dfa']:
    if nl_feat in df_check.columns:
        vals = df_check[nl_feat].dropna()
        print(f'  {nl_feat.split("__")[-1]:<20}: min={vals.min():.3f}  median={vals.median():.3f}  max={vals.max():.3f}  NaN={df_check[nl_feat].isna().sum()}')

In [ ]:
# Visualização: distribuições de features selecionadas por classe
fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(4, 4, figure=fig, hspace=0.5, wspace=0.4)

# Features para plotar: 1 representante por família
features_plot = [
    # (coluna, título)
    ('EEG-C3__time__RMS',              'EEG-C3  RMS (tempo)'),
    ('EEG-C3__time__ZCR',              'EEG-C3  ZCR (tempo)'),
    ('EEG-C3__time__hjorth_mobility',  'EEG-C3  Hjorth Mobility'),
    ('EEG-C3__time__hjorth_complexity','EEG-C3  Hjorth Complexity'),
    ('EEG-C3__freq__band_alpha',       'EEG-C3  Band Power Alpha'),
    ('EEG-C3__freq__band_beta',        'EEG-C3  Band Power Beta'),
    ('EEG-C3__freq__median_freq',      'EEG-C3  Frequência Mediana'),
    ('EMG-RTA__freq__band_low',        'EMG-RTA Band Power Low'),
    ('EEG-C3__tfreq__stft_alpha_t0',   'EEG-C3  STFT alpha t0'),
    ('EEG-C3__tfreq__stft_beta_t2',    'EEG-C3  STFT beta t2'),
    ('EEG-C3__tfreq__dwt_D4_energy',   'EEG-C3  DWT D4 energy'),
    ('EEG-C3__tfreq__dwt_A4_energy',   'EEG-C3  DWT A4 energy'),
    ('EEG-C3__nonlin__sample_entropy', 'EEG-C3  SampEn'),
    ('EEG-C3__nonlin__approx_entropy', 'EEG-C3  ApEn'),
    ('EEG-C3__nonlin__dfa',            'EEG-C3  DFA α'),
    ('LShank-ACCZ__freq__band_movimento', 'LShank-ACCZ Band Movimento'),
]

mask0 = df_check['task_label'] == 0.0
mask1 = df_check['task_label'] == 1.0

cores = {0: 'steelblue', 1: 'tomato'}
for plot_i, (col, titulo) in enumerate(features_plot):
    row, col_gs = divmod(plot_i, 4)
    ax = fig.add_subplot(gs[row, col_gs])

    if col not in df_check.columns:
        ax.text(0.5, 0.5, 'ausente', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(titulo, fontsize=7)
        continue

    for lbl, mask, cor in [(0, mask0, 'steelblue'), (1, mask1, 'tomato')]:
        vals = df_check.loc[mask, col].dropna()
        if len(vals) == 0:
            continue
        # Clipar outliers extremos para melhor visualização
        p1, p99 = vals.quantile([0.01, 0.99])
        vals_clip = vals.clip(p1, p99)
        ax.hist(vals_clip, bins=30, alpha=0.55, color=cor,
                label=f'label={lbl}', density=True)

    ax.set_title(titulo, fontsize=7)
    ax.set_xlabel('')
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2)
    if plot_i == 0:
        ax.legend(fontsize=6)

fig.suptitle('Distribuição das features por classe (Normal vs FoG)\n'
             'Distribuições sobrepostas indicam menor poder discriminatório; '
             'separadas indicam maior poder.',
             fontsize=10)
plt.show()

In [ ]:
# Estatística de separabilidade: Mann-Whitney U e Cohen's d por feature selecionada
from scipy.stats import mannwhitneyu

print('Separabilidade Normal vs FoG (features selecionadas):')
print(f'{"Feature":<50} {"U-stat":>10} {"p-valor":>12} {"Cohen d":>10}')
print('-' * 84)

for col, titulo in features_plot:
    if col not in df_check.columns:
        continue
    v0 = df_check.loc[mask0, col].dropna().values
    v1 = df_check.loc[mask1, col].dropna().values
    if len(v0) < 5 or len(v1) < 5:
        continue

    try:
        stat, p = mannwhitneyu(v0, v1, alternative='two-sided')
    except:
        continue

    # Cohen's d
    pooled_std = np.sqrt((np.var(v0, ddof=1) + np.var(v1, ddof=1)) / 2)
    cohd = (np.mean(v1) - np.mean(v0)) / (pooled_std + 1e-10)

    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ' ns'))
    print(f'{col:<50} {stat:>10.0f} {p:>12.2e} {cohd:>9.3f}  {sig}')

print()
print('Legenda: * p<0.05  ** p<0.01  *** p<0.001  (Mann-Whitney U, bilateral)')
print('Cohen d: |d|<0.2 pequeno, 0.2-0.5 médio, 0.5-0.8 grande, >0.8 muito grande')

In [ ]:
print('=' * 65)
print('EXTRAÇÃO 2 — FEATURES — RESUMO FINAL')
print('=' * 65)
print()
print('Entrada:')
print(f'  segments.npz  → X {X.shape}  y {y.shape}')
print()
print('Features extraídas:')
print(f'  Domínio do tempo    : {len(feat_time):>5,}')
print(f'    RMS, MAV, VAR, ZCR: {4*N_CH:>5,}  ({N_CH} canais × 4)')
print(f'    Hjorth (EEG)       : {3*n_eeg:>5,}  ({n_eeg} canais × 3)')
print(f'  Domínio da frequência: {len(feat_freq):>5,}')
print(f'  Tempo-frequência (STFT): {len(feat_stft):>4,}')
print(f'  Tempo-frequência (DWT) : {len(feat_dwt):>4,}')
print(f'  Não-lineares         : {len(feat_nonlin):>5,}  ({len(canais_nl_presentes)} canais × 3 métricas)')
print(f'  ─────────────────────────────────────')
print(f'  TOTAL FEATURES       : {n_features:>5,}')
print()
print('Saída:')
print(f'  ✅ {FEATURES_PATH.name}')
print(f'     {N_WIN:,} janelas × {n_features:,} features  ({FEATURES_PATH.stat().st_size/1e6:.1f} MB)')
print()
print('Distribuição de labels:')
print(f'  Normal (0): {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)')
print(f'  FoG    (1): {(y==1).sum():,}  ({(y==1).mean()*100:.1f}%)')
print()
print('Próximo passo:')
print('  extração_3_normalizacao.ipynb')
print('  → Tratar NaN (imputação por mediana ou remoção de canal)')
print('  → Normalizar features (StandardScaler por canal, fit no treino)')
print('  → Produzir features_normalized.parquet')
print('=' * 65)